# Dashboard (Phase 5: anymap-ts)

This notebook subscribes to agent topics and visualizes live person state on a map.

Phase 5 scope:
- Subscribe to person state, camera decisions, and entry events
- Render live markers with white/green/red colors
- Track occupancy from entered/exited events
- Keep dashboard read-only (no control decisions)

In [ ]:
# Cell purpose: import libraries and load shared project configuration
from __future__ import annotations

import json
import time
import sys
import os
from pathlib import Path
from collections import defaultdict

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    src_dir = candidate / "src"
    if (src_dir / "simulated_city").exists():
        if str(src_dir) not in sys.path:
            sys.path.insert(0, str(src_dir))
        break

os.environ.setdefault("SIMCITY_MQTT_PROFILES", "mqtthq")

from anymap_ts import Map
from simulated_city.config import load_config
import simulated_city.mqtt as mqtt

cfg = load_config()
sim_cfg = cfg.simulation
mqtt_cfg = cfg.mqtt

CENTER_LAT = sim_cfg.center_lat if sim_cfg else 55.6479
CENTER_LON = sim_cfg.center_lon if sim_cfg else 12.0417

print(f"Loaded config. MQTT base topic: {mqtt_cfg.base_topic}")
print(f"Dashboard center: lat={CENTER_LAT}, lon={CENTER_LON}")

Loaded config. MQTT base topic: simulated-city/stadium
Dashboard center: lat=55.6479, lon=12.0417


In [2]:
# Cell purpose: create the anymap-ts map widget used for live visualization
map_view = Map(center=(CENTER_LON, CENTER_LAT), zoom=16.5, height="650px", width="100%")
map_view.add_basemap("OpenStreetMap.Mapnik")

COLOR_HEX = {
    "white": "#FFFFFF",
    "green": "#2E7D32",
    "red": "#C62828",
}

print("Map created with OpenStreetMap.Mapnik basemap.")
map_view

Map created with OpenStreetMap.Mapnik basemap.


In [3]:
# Cell purpose: initialize MQTT client, topics, and dashboard state stores
client = mqtt.connect_mqtt(mqtt_cfg, client_id_suffix="dashboard-phase5")

person_state_topic = f"{mqtt_cfg.base_topic}/person/state"
camera_decision_topic = f"{mqtt_cfg.base_topic}/camera/decision"
entry_event_topic = f"{mqtt_cfg.base_topic}/entry/event"

latest_people: dict[str, dict] = {}
marker_ids: dict[str, str] = {}
decision_counts = defaultdict(int)
inside_count = 0
message_counts = defaultdict(int)
last_render_ts = time.time()

print(f"Connected to MQTT broker at {mqtt_cfg.host}:{mqtt_cfg.port}")
print(f"Subscribing to: {person_state_topic}")
print(f"Subscribing to: {camera_decision_topic}")
print(f"Subscribing to: {entry_event_topic}")

Error connecting to MQTT broker: [Errno 61] Connection refused


ConnectionRefusedError: [Errno 61] Connection refused

In [4]:
# Cell purpose: define subscriber callbacks to update markers, decisions, and occupancy
def _safe_color(value: str) -> str:
    lowered = (value or "white").lower()
    return lowered if lowered in COLOR_HEX else "white"

def _render_person_marker(person_payload: dict) -> None:
    person_id = str(person_payload.get("person_id", "unknown"))
    lat = person_payload.get("lat")
    lon = person_payload.get("lon")
    if lat is None or lon is None:
        return

    color_name = _safe_color(person_payload.get("color", "white"))
    marker_name = f"person-{person_id}"

    previous_marker_id = marker_ids.get(person_id)
    if previous_marker_id:
        try:
            map_view.remove_marker(previous_marker_id)
        except Exception:
            pass

    popup = (
        f"person_id={person_id}<br>"
        f"state={person_payload.get('state', 'unknown')}<br>"
        f"color={color_name}<br>"
        f"entry={person_payload.get('target_entry_id', 'unknown')}"
    )

    marker_id = map_view.add_marker(
        float(lon),
        float(lat),
        name=marker_name,
        color=COLOR_HEX[color_name],
        popup=popup,
    )
    marker_ids[person_id] = marker_id

def _print_dashboard_status() -> None:
    print(
        "dashboard status -> "
        f"people={len(latest_people)} inside_count={inside_count} "
        f"camera_allowed={decision_counts['allowed']} "
        f"camera_denied={decision_counts['denied']}"
    )

def on_message(client, userdata, msg):
    global inside_count, last_render_ts

    topic = msg.topic
    message_counts[topic] += 1

    try:
        payload = json.loads(msg.payload.decode())
    except Exception:
        return

    if topic == person_state_topic:
        person_id = str(payload.get("person_id", "unknown"))
        latest_people[person_id] = payload
        _render_person_marker(payload)
    elif topic == camera_decision_topic:
        allowed = bool(payload.get("allowed", False))
        decision_counts["allowed" if allowed else "denied"] += 1
    elif topic == entry_event_topic:
        event_type = str(payload.get("event_type", "")).lower()
        if event_type == "entered":
            inside_count += 1
        elif event_type == "exited":
            inside_count = max(0, inside_count - 1)

    now = time.time()
    if now - last_render_ts >= 1.0:
        _print_dashboard_status()
        last_render_ts = now

client.on_message = on_message
client.subscribe(person_state_topic, qos=0)
client.subscribe(camera_decision_topic, qos=0)
client.subscribe(entry_event_topic, qos=0)

print("Dashboard subscriptions active.")

NameError: name 'client' is not defined

In [ ]:
# Cell purpose: keep dashboard listener active and print periodic lightweight heartbeat
print("Dashboard running. Press Interrupt to stop.")
while True:
    time.sleep(5)
    print(
        "heartbeat -> "
        f"person_msgs={message_counts[person_state_topic]} "
        f"decision_msgs={message_counts[camera_decision_topic]} "
        f"entry_msgs={message_counts[entry_event_topic]} "
        f"inside_count={inside_count}"
    )